# SVM against random forest on Covertype — at an equal training budget

581,012 rows, 54 features, 7 classes, roughly 100:1 imbalance.

An RBF SVM **cannot** be trained on 581,012 rows. Kernel training is between quadratic and
cubic in the row count, and the kernel matrix alone is about 2,700 GB. So every published
version of this comparison subsamples for the SVM.

The mistake is to subsample for the SVM, train the forest on everything, and put both
accuracies in the same table. That table compares two different experiments.

This notebook does three things instead:

1. gives every model the **same** training budget, as an explicit parameter;
2. reports accuracy next to the numbers that survive a 100:1 imbalance;
3. prices the preprocessing decision that is usually invisible.

Everything imports `src/covertype/`. The measurement half is standard library.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))

from covertype import synthetic
from covertype.baseline import MajorityClassifier, NearestCentroid
from covertype.data import CLASS_NAMES, invert_one_hot, load_csv
from covertype.experiment import (
    cost_of_a_point,
    evaluate,
    learning_curve,
    ranking_disagreement,
    table,
)
from covertype.metrics import majority_accuracy
from covertype.models import SPECS, build, feasible
from covertype.sampling import split, stratified_sample, uniform_sample

## The dataset's shape is the whole problem

Two classes are 85% of the rows; the rarest is under half a percent.

The default below is a generated dataset with the same shape — seven classes, ~100:1
imbalance, features spanning four orders of magnitude — so every claim in this notebook is
reproducible without a 75 MB download. `load_csv` runs the identical experiment on the
real thing; see the last section.

In [ ]:
dataset = synthetic.generate(30_000, seed=0)
counts = dataset.class_counts()

print(
    f"{len(dataset):,} rows, {dataset.n_features} features, "
    f"imbalance {dataset.imbalance_ratio():.0f}:1\n"
)
for label, name in CLASS_NAMES.items():
    print(f"  {label} {name:<20}{counts[label]:>8,}{counts[label] / len(dataset):>8.2%}")

print(f"\nmajority-class accuracy: {majority_accuracy(list(dataset.y)):.4f}")

## Why the SVM run has to be refused, with the arithmetic

A guard that fires after four hours of swapping is not a guard.

In [ ]:
for rows in (20_000, 100_001, 581_012):
    ok, why = feasible("svm-rbf", rows)
    print(f"{rows:>9,}  {'ok' if ok else 'REFUSED'}  {why}")

## Demonstration 1 — a model that predicts nothing

Predict "lodgepole pine" for every row. Six of the seven classes are never predicted at
all, and the accuracy column calls it a respectable model.

In [ ]:
sample = split(dataset, train_size=4000, test_size=4000, seed=0)

majority = evaluate(lambda: MajorityClassifier(), sample, name="majority")
print(majority.report.summary())
print(
    f"\n{len(majority.report.abandoned_classes)} of 7 classes never predicted, "
    f"for {majority.accuracy:.1%} accuracy."
)

`abandoned_classes` is a separate flag rather than something inferred from an F1 of zero,
because refusing to use a class is a *decision* the accuracy column cannot show — and on
this dataset it is almost free.

## Demonstration 2 — accuracy and macro-F1 rank models in opposite order

In [ ]:
centroid = evaluate(lambda: NearestCentroid(), sample, name="centroid")
print(table([majority, centroid]))
print()
print(ranking_disagreement([majority, centroid]))

This is not a subtlety. It is the reason a comparison table on this dataset sorted by
accuracy is a table sorted by how well each model predicts lodgepole pine.

**Balanced accuracy** is the mean per-class recall — what accuracy would be if every class
were equally common. **Macro-F1** additionally punishes a model that reaches recall by
predicting a class indiscriminately.

## Demonstration 3 — the preprocessing decision costs more than the model choice

An RBF kernel measures distance. On unstandardised Covertype,
`Horizontal_Distance_To_Roadways` spans 0–7,000 and `Slope` spans 0–66, so the kernel is
effectively one-dimensional. A random forest has no such problem: a tree split is
invariant to monotone rescaling of a feature.

So an unscaled SVM compared against a forest is a broken configuration compared against a
working one — and the gap is larger than anything between the model families.

In [ ]:
scaled = evaluate(lambda: NearestCentroid(), sample, scale=True, name="centroid+scaled")
print(table([centroid, scaled]))
print(
    f"\nstandardising: {scaled.accuracy - centroid.accuracy:+.4f} accuracy, "
    f"{scaled.macro_f1 - centroid.macro_f1:+.4f} macro-F1 — same model, same data."
)

In [ ]:
# Which is why the SVM factories are pipelines and the forest is not.
for kind, spec in SPECS.items():
    print(
        f"{kind:<16}{spec.name:<18}needs scaling: {spec.needs_scaling!s:<6}"
        f"training cost: {spec.complexity}"
    )

## Demonstration 4 — a uniform subsample can delete a class

Cover type 4 is under 0.5% of the rows. Stratifying is not about fairness here; it is
about the experiment being runnable at all.

In [ ]:
rarest = synthetic.rarest_label()
for name, draw in (("uniform", uniform_sample), ("stratified", stratified_sample)):
    found = [draw(dataset, 500, seed=s).y.count(rarest) for s in range(10)]
    print(f"{name:<12}cover type {rarest} in a 500-row draw: {found}")

## The comparison itself: equal budgets, and what the next 4x of data buys

In [ ]:
sizes = [500, 2000, 8000]
runs = learning_curve(
    dataset,
    lambda: NearestCentroid(),
    sizes,
    test_size=4000,
    scale=True,
    name="centroid+scaled",
) + learning_curve(
    dataset, lambda: MajorityClassifier(), sizes, test_size=4000, name="majority"
)
print(table(runs))
print()
for name in ("centroid+scaled", "majority"):
    curve = [r for r in runs if r.model == name]
    print(f"{name}: {cost_of_a_point(curve)}")

The test set is held at a fixed size across the curve. Letting it grow with the training
budget changes the noise level from point to point, and the curve stops being a curve in
one variable.

## The feature-ranking step that means nothing

The usual EDA for this dataset collapses the 40 one-hot soil columns into a single integer
and ranks features by Pearson correlation with `Cover_Type`.

Both halves are broken. Soil type 17 is not "between" 16 and 18, and a correlation
coefficient against a *nominal* 7-class label has a value that depends entirely on the
arbitrary integers assigned to the categories — renumber them and the "most correlated
feature" changes.

`invert_one_hot` exists for contingency tables, refuses blocks that are not actually
one-hot, and says all of this in its docstring.

In [ ]:
codes = (
    invert_one_hot(dataset, "Soil")
    if any(c.startswith("Soil") for c in dataset.columns)
    else None
)
print("the synthetic dataset has no one-hot blocks; on the real file:")
print("  invert_one_hot(real, 'Soil_Type')  ->  [1, 29, 12, ...]   (for crosstabs only)")

## Running it on the real dataset

```bash
pip install -r requirements.txt
python -m covertype --uci --models svm-rbf,random-forest --train-sizes 5000,20000
```

In [ ]:
csv = Path("covtype.csv")
if csv.exists():
    real = load_csv(csv)
    print(
        f"rows {len(real):,}  features {real.n_features}  "
        f"imbalance {real.imbalance_ratio():.0f}:1"
    )
    real_sample = split(real, train_size=5000, test_size=5000, seed=0)
    real_runs = [
        evaluate(lambda: build("svm-rbf", seed=0), real_sample, name="svm-rbf"),
        evaluate(
            lambda: build("random-forest", seed=0), real_sample, name="random-forest"
        ),
    ]
    print(table(real_runs))
    print()
    print(ranking_disagreement(real_runs))
else:
    print("covtype.csv not found — see the README, or use --uci.")

## Conclusion

The exercise this notebook replaces asked "which model obtains better classification
results". On this dataset that question cannot be answered from an accuracy column, and
three things have to be settled before it can be asked at all:

1. **the same training budget for both models** — otherwise the answer is "the one that
   saw more data";
2. **scaling for the model that needs it** — worth more than the model choice itself;
3. **a metric that survives 100:1 imbalance** — accuracy ranks a model that predicts one
   class forever above one that predicts all seven.

The comparison that remains is a real engineering trade: the forest trains in `n log n`
and handles the full dataset; the SVM is quadratic-to-cubic and is confined to a
subsample, where the question becomes whether its accuracy *at that budget* justifies the
ceiling on how much data it can ever use.